In [1]:
import os
import warnings 
import random 
from zipfile import ZipFile 
from pathlib import Path 

import pandas as pd 
import sklearn 
import numpy as np

import matplotlib.pyplot as plt 
from tqdm.auto import tqdm 
from torchinfo import summary

import torch
from torch import nn 
from torch .utils.data import DataLoader, Dataset 
from torchvision import transforms 
from torchvision.transforms import v2
from sklearn.model_selection import StratifiedKFold 

warnings.filterwarnings("ignore")

In [14]:
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

In [7]:
device = torch.device("mps"
if torch.backends.mps.is_available()
else "cuda"
if torch.cuda.is_available()
else "cpu")

print(f"Device: '{device}'")

Device: 'mps'


In [10]:
SEED = 42

In [12]:
def seed_everything(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic=(True)
    os.environ["PYTHONHASHSEED"] = str(seed)

In [13]:
seed_everything(SEED)

In [16]:
data_path = Path("data")
file_path = data_path / "digit_recognizer"
file_path_zip = data_path / "digit-recognizer.zip"

with ZipFile(file_path_zip, "r") as rep_zip:
    print(f"Unzipping digit recognizer data in {file_path}...")
    rep_zip.extractall(file_path)

Unzipping digit recognizer data in data/digit_recognizer...


In [19]:
# train_df = pd.read_csv("/kaggle/input/competitions/digit-recognizer/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/digit-recognizer/test.csv")

train_dir = file_path / "train.csv"
test_dir = file_path / "test.csv"

train_df = pd.read_csv(train_dir)
test_df = pd.read_csv(test_dir)

X_raw = train_df.drop(columns=["label"]).values 
y_raw = train_df["label"].values
X_test_raw = test_df.values 

X_train_all = (
    torch.tensor(X_raw, dtype=torch.float32).reshape(-1, 1, 28, 28) / 255.0
)
y_train_all = (
    torch.tensor(y_raw, dtype=torch.long)
)
X_test_all = (
    torch.tensor(X_test_raw, dtype=torch.float32).reshape(-1, 1, 28, 28) / 255.0
)

print(f"X Train tensor shape: {X_train_all.shape}")
print(f"Y Train shape: {y_train_all.shape}")
print(f"X Test shape: {X_test_all.shape}") 

X Train tensor shape: torch.Size([42000, 1, 28, 28])
Y Train shape: torch.Size([42000])
X Test shape: torch.Size([28000, 1, 28, 28])
